# Как VAE видит фото кошки

Готовый автоэнкодер из Stable Diffusion сжимает картинку 512x512x3 в латент 64x64x4 и восстанавливает ее обратно.
Диффузия будет работать только с латентами, поэтому все, что VAE теряет здесь, модель потом не сможет нарисовать.
Ноутбук показывает, что именно теряется на фото кошки и какое разрешение стоит брать.

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import AutoencoderKL

from train import center_crops, load_images

VAE_NAME = 'stabilityai/sd-vae-ft-mse'    # веса уже лежат в ~/.cache/huggingface
DATA_DIR = 'dataset_unique'
SIZE = 512                                # сторона квадратного центрального кропа
BATCH = 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

vae = AutoencoderKL.from_pretrained(VAE_NAME, local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)

SCALE = vae.config.scaling_factor
F_DOWN = 2 ** (len(vae.config.block_out_channels) - 1)
n_par = sum(p.numel() for p in vae.parameters())
print(f'{VAE_NAME}: {n_par / 1e6:.1f} M параметров, сжатие в {F_DOWN} раз, '
      f'{vae.config.latent_channels} канала, scaling_factor {SCALE}, устройство {DEVICE}')

In [ ]:
@torch.no_grad()
def encode(x):
    """(B, 3, S, S) в [-1, 1] -> среднее и log-дисперсия латента, оба (B, 4, S/8, S/8)."""
    d = vae.encode(x.to(DEVICE)).latent_dist
    return d.mean, d.logvar


@torch.no_grad()
def decode(z):
    """Латент без домножения на scaling_factor -> картинка в [-1, 1]."""
    return vae.decode(z).sample.clamp(-1, 1)


def to_uint8(x):
    """[-1, 1] -> uint8 на CPU, так 395 картинок занимают 300 МБ, а не 6 ГБ."""
    return ((x + 1) * 127.5).round().clamp(0, 255).byte().cpu()


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) для imshow."""
    return img.permute(1, 2, 0).numpy()


def psnr(a, b):
    """PSNR в дБ на каждую картинку, a и b: (B, 3, H, W) uint8. Больше = лучше."""
    mse = ((a.float() - b.float()) ** 2).mean((1, 2, 3))
    return 10 * torch.log10(255 ** 2 / mse)

## Все фото через VAE

Берем среднее латента, без случайной добавки: так виден лучший результат, на который способен VAE.

In [ ]:
t0 = time.time()
files, imgs = load_images(DATA_DIR, SIZE)
print(f'фото {len(imgs)}, загрузка {time.time() - t0:.1f} с')

t0 = time.time()
X, R, Z, S = [], [], [], []
for i in range(0, len(imgs), BATCH):
    x = center_crops(imgs[i:i + BATCH], SIZE)
    mean, logvar = encode(x)
    X.append(to_uint8(x))
    R.append(to_uint8(decode(mean)))
    Z.append(mean.cpu())
    S.append(torch.exp(0.5 * logvar).cpu())
X, R, Z, S = torch.cat(X), torch.cat(R), torch.cat(Z), torch.cat(S)
P = psnr(X, R).numpy()

print(f'туда и обратно за {time.time() - t0:.1f} с, {(time.time() - t0) / len(imgs) * 1e3:.0f} мс на фото')
print(f'PSNR: медиана {np.median(P):.1f} дБ, 10% квантиль {np.quantile(P, 0.1):.1f}, '
      f'мин {P.min():.1f}, макс {P.max():.1f}')

In [ ]:
# Диффузии нужны латенты с дисперсией около 1, поэтому проверяем множитель на своих фото
mean_c = Z.mean((0, 2, 3))
std_c = Z.std((0, 2, 3))
print('канал     среднее   ст.откл')
for c in range(Z.shape[1]):
    print(f'{c:5d} {mean_c[c].item():10.3f} {std_c[c].item():9.3f}')

print(f'\nст.откл по всем каналам {Z.std().item():.3f}, значит множитель для своих фото {1 / Z.std().item():.5f} '
      f'(у модели {SCALE})')
print(f'собственный шум латента: средняя sigma {S.mean().item():.4f}, '
      f'это {S.mean().item() / Z.std().item() * 100:.1f}% от разброса латентов')

## Худшие, средние и лучшие фото

Строки: оригинал, результат VAE, ошибка, усиленная в 4 раза.
PSNR считается по всему кадру, поэтому худшими оказываются фото с мелкой текстурой фона, а не с плохо восстановленной кошкой.

In [ ]:
order = np.argsort(P)
mid = len(order) // 2
show = [*order[:2], *order[mid - 1:mid + 1], *order[-2:]]

fig, axs = plt.subplots(3, len(show), figsize=(3.2 * len(show), 9.8))
for j, k in enumerate(show):
    err = (X[k].float() - R[k].float()).abs().mean(0).numpy() / 255
    axs[0, j].imshow(to_np(X[k]))
    axs[1, j].imshow(to_np(R[k]))
    axs[2, j].imshow((err * 4).clip(0, 1), cmap='magma', vmin=0, vmax=1)
    axs[0, j].set_title(f'№{k}, PSNR {P[k]:.1f} дБ')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Обзор

Номера над картинками нужны для списка `ZOOMS` в следующей ячейке. Линии идут через 128 пикселей.

In [ ]:
START = 0
N_SHOW = 48
COLS = 8

idx = list(range(START, min(START + N_SHOW, len(X))))
rows = (len(idx) + COLS - 1) // COLS
fig, axs = plt.subplots(rows, COLS, figsize=(2.6 * COLS, 2.8 * rows))
for ax in axs.flat:
    ax.axis('off')
for ax, k in zip(axs.flat, idx):
    ax.axis('on')
    ax.imshow(to_np(X[k]))
    ax.set_title(f'№{k}', fontsize=10)
    ax.set_xticks([128, 256, 384], labels=[])
    ax.set_yticks([128, 256, 384], labels=[])
    ax.grid(color='w', alpha=0.4, lw=0.5)
    ax.tick_params(length=0)
plt.tight_layout()
plt.show()

## Морда и шерсть крупно

В `ZOOMS` записаны участки (номер фото, y, x), координаты центра в пикселях картинки 512x512.
Строки: оригинал, результат VAE, ошибка, усиленная в 4 раза.

In [ ]:
ZOOMS = [(0, 200, 190), (1, 185, 220), (8, 150, 218), (12, 150, 268), (28, 205, 300), (37, 163, 235), (0, 190, 420), (374, 100, 100)]
WIN = 128     # сторона участка в пикселях


def crop(img, y, x, win):
    """Квадрат win x win с центром (y, x), сдвинутый так, чтобы не вылезти за край."""
    _, H, W = img.shape
    top = min(max(y - win // 2, 0), H - win)
    left = min(max(x - win // 2, 0), W - win)
    return img[:, top:top + win, left:left + win]


fig, axs = plt.subplots(3, len(ZOOMS), figsize=(3.2 * len(ZOOMS), 9.8))
for j, (k, y, x) in enumerate(ZOOMS):
    a, b = crop(X[k], y, x, WIN), crop(R[k], y, x, WIN)
    err = (a.float() - b.float()).abs().mean(0).numpy() / 255
    axs[0, j].imshow(to_np(a), interpolation='nearest')
    axs[1, j].imshow(to_np(b), interpolation='nearest')
    axs[2, j].imshow((err * 4).clip(0, 1), cmap='magma', vmin=0, vmax=1, interpolation='nearest')
    axs[0, j].set_title(f'№{k}, участок {WIN}x{WIN}')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Что лежит в латенте

Те же 512x512, но глазами диффузии: 4 карты 64x64. Цветная картинка в латенте все еще узнается,
поэтому U-Net и здесь работает со свертками по пространству.

In [ ]:
IDX = 36

z = Z[IDX] * SCALE     # масштаб, с которым латент увидит диффузия
lim = z.abs().quantile(0.995).item()

fig, axs = plt.subplots(1, 6, figsize=(21, 3.8))
axs[0].imshow(to_np(X[IDX]))
axs[0].set_title('оригинал, 512x512x3')
for c in range(z.shape[0]):
    axs[c + 1].imshow(z[c].numpy(), cmap='gray', vmin=-lim, vmax=lim, interpolation='nearest')
    axs[c + 1].set_title(f'канал {c}, 64x64')
axs[5].imshow(to_np(R[IDX]))
axs[5].set_title(f'декодер, PSNR {P[IDX]:.1f} дБ')
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Какое разрешение выбрать

Одни и те же фото на разных размерах, только те, у которых короткая сторона не меньше 1024: их не приходится растягивать.
PSNR между размерами сравнивать нельзя: на меньшем размере те же детали упакованы плотнее, и это другая задача.
Смотреть надо на участки на следующем рисунке: что остается от морды при латенте 32x32 (256) и 64x64 (512).

In [ ]:
SIZES = [256, 512, 768, 1024]
N_TEST = 32

short = np.array([min(Image.open(p).size) for p in files])
big = np.flatnonzero(short >= 1024)
test = set(big[np.linspace(0, len(big) - 1, N_TEST).round().astype(int)].tolist())
zooms_big = [z for z in ZOOMS if short[z[0]] >= 1024]
test |= {z[0] for z in zooms_big}
print(f'короткая сторона >= 1024 у {len(big)} фото из {len(files)}, в тесте {len(test)}')

names = {files[k].name for k in test}
skip = {f.name for f in files} - names

res = {}
print('размер   латент       PSNR, медиана   мс на фото')
for size in SIZES:
    fs, ims = load_images(DATA_DIR, size, exclude=skip)
    bs = max(2, int(BATCH * (512 / size) ** 2))
    t0 = time.time()
    xs, rs = [], []
    for i in range(0, len(ims), bs):
        x = center_crops(ims[i:i + bs], size)
        xs.append(to_uint8(x))
        rs.append(to_uint8(decode(encode(x)[0])))
    dt = time.time() - t0
    xs, rs = torch.cat(xs), torch.cat(rs)
    res[size] = {'x': dict(zip([f.name for f in fs], xs)), 'r': dict(zip([f.name for f in fs], rs))}
    lat = size // F_DOWN
    print(f'{size:6d}   {lat:3d}x{lat:<3d}x4 {np.median(psnr(xs, rs).numpy()):12.1f} дБ {dt / len(fs) * 1e3:10.0f}')
    torch.cuda.empty_cache()

In [ ]:
# Один и тот же участок: слева оригинал на 1024, дальше результат VAE на каждом размере
rows = len(zooms_big)
fig, axs = plt.subplots(rows, len(SIZES) + 1, figsize=(3.4 * (len(SIZES) + 1), 3.5 * rows), squeeze=False)
for i, (k, y, x) in enumerate(zooms_big):
    name = files[k].name
    s0 = SIZES[-1] / SIZE
    axs[i, 0].imshow(to_np(crop(res[SIZES[-1]]['x'][name], int(y * s0), int(x * s0), int(WIN * s0))),
                     interpolation='nearest')
    axs[i, 0].set_title(f'№{k}, оригинал {SIZES[-1]}')
    for j, size in enumerate(SIZES):
        s = size / SIZE
        axs[i, j + 1].imshow(to_np(crop(res[size]['r'][name], int(y * s), int(x * s), int(WIN * s))),
                             interpolation='nearest')
        axs[i, j + 1].set_title(f'VAE, {size}')
for ax in axs.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()